# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is primarily a **ranking / scoring** task: "which pages should an editor review first?" —
not a flat yes/no call on any single page, but an ordering across the whole inventory.

Under the hood I build it as **binary classification** — predict the probability that a page is
declining — and then use that predicted probability as the score to rank pages by. This matches
the framing guide's mapping: "which ones first?" → ranking/scoring, target = a priority score,
metric = precision@K. Classification is the mechanism; ranking is the actual task.

In [15]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} columns")
print("Task type: ranking/scoring, implemented as binary classification whose predicted")
print("probability becomes the priority score for the editor's queue.")


Loaded 30,000 rows, 44 columns
Task type: ranking/scoring, implemented as binary classification whose predicted
probability becomes the priority score for the editor's queue.


In [16]:
subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_all.py'], returncode=0)

## 2. Target or proxy

**Target:** `is_declining_label` — defined as `trend_direction == "down"`.

**Where it comes from:** `trend_direction` is computed from `trend_pct`, which itself compares
`impressions_last_30d` against `impressions_prev_30d` — a real, observed change in a page's own
traffic over time. So the label is an **observed outcome** (impressions genuinely fell), not
someone's subjective opinion about the page.

**But it's still a proxy, and I have to be honest about that:** "impressions declined" is not the
same thing as "this page needs a refresh" or "a refresh would fix it." A page can decline for
reasons a content refresh won't touch (seasonality, a SERP feature change, a competitor's new
page). The label tells me *something changed*, not *what to do about it* or *whether refreshing
helps* — the model's job is to help editors find candidates worth a human look, not to replace
that judgment.

In [17]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df["trend_direction"].value_counts())
print()
print(f"is_declining_label rate: {df['is_declining_label'].mean():.1%} "
      f"({df['is_declining_label'].sum():,} of {len(df):,} rows)")



trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label rate: 54.2% (16,262 of 30,000 rows)


## 3. Success metric

**Primary metric: Precision@50.**

An editor only reviews a fixed number of pages per week — not the whole ranked list. So what
matters is: *of the top 50 pages the model says to look at first, how many are actually
declining?* That's exactly Precision@K, evaluated at the K an editor can realistically act on.

I'm not leading with plain accuracy or recall, because:
- **Accuracy** is misleading here — the base rate is already ~54% declining, so a model can look
  "accurate" while still being useless for prioritization.
- **Recall** (catching every declining page) doesn't match how the output gets used — nobody is
  reviewing all 22,000 visible pages regardless of what the model says, so a metric that only
  rewards catching everything ignores the real constraint (editor time).

Precision@50 is the one number I can defend to an editor: "here's how often the top of your
queue is actually worth your time."

In [18]:
import json

res = json.load(open("outputs/model_results.json")) if os.path.exists("outputs/model_results.json") else None
if res:
    base = res["baseline"]["baseline_precision_at_50"]
    rf   = res["models"]["random_forest"]["precision_at_50"]
    print(f"From my earlier pipeline run (notebook 01):")
    print(f"  Hand-written rule  Precision@50: {base:.3f}")
    print(f"  Random forest      Precision@50: {rf:.3f}")
else:
    print("outputs/model_results.json not found yet — re-run scripts/run_all.py to regenerate it.")

From my earlier pipeline run (notebook 01):
  Hand-written rule  Precision@50: 0.240
  Random forest      Precision@50: 0.740


## 4. The unit of analysis, as a real dataframe

**One row = one piece of content (a page) belonging to one client**, described by its trailing
90-day search and engagement metrics. `content_id` + `client_id` together identify the row —
each is a pseudonym, used only for grouping/joining/splitting, never as a feature.

Below is the actual slice this lane works from: visible pages (enough impressions to matter),
with the columns that would feed the model plus the target column.

In [19]:
visible = df[df["impressions_90d"] >= 100].copy()

lane_cols = [
    "content_id", "client_id",                       # identifiers (never features)
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "content_age_days", "freshness_tier", "word_count",
    "content_type", "position_tier",
    "is_declining_label",                             # target
]

print(f"Unit of analysis: one row = one page (content_id) for one client (client_id).")
print(f"Lane slice: {len(visible):,} visible pages\n")
visible[lane_cols].head()



Unit of analysis: one row = one page (content_id) for one client (client_id).
Lane slice: 22,006 visible pages



,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,freshness_tier,word_count,content_type,position_tier,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,187,0-30,3221.0,keyword article,striking,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,445,0-30,2481.0,keyword article,page_3_5,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,141,0-30,3515.0,keyword article,page_3_5,1
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,463,0-30,NaN,keyword article,page_1,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,263,0-30,2803.0,keyword article,page_3_5,1


## 5. Why ML beats a fixed rule here

I already tested this directly in notebook 01: a transparent hand-written rule (a simple
threshold on staleness + declining signal) scored **Precision@50 = 0.240**, while a random forest
trained on the same data scored **0.740** — about **3.1x** better at the exact same job.

That gap is the evidence, not a guess. It happens because no single signal cleanly separates
"needs a refresh" from "fine as is":
- A page can be old (stale) but still growing — freshness alone over-flags it.
- A page can decline for reasons unrelated to content quality (seasonality, a SERP layout
  change) — decline alone over-flags it too.
- Position, impressions, word count, and freshness interact — e.g. a page at position 4 that's
  losing impressions is a different risk than a page at position 9 doing the same.

A fixed if/else rule has to pick one or two of these and threshold them by hand. A model can
weigh all of them together and learn where the real boundary sits — which is exactly why the
random forest outperformed the hand-written rule on data it had never seen (client-holdout
split, so no leakage).

In [20]:
if res:
    print(f"Hand-written rule  Precision@50: {res['baseline']['baseline_precision_at_50']:.3f}")
    print(f"Random forest       Precision@50: {res['models']['random_forest']['precision_at_50']:.3f}")
    ratio = res['models']['random_forest']['precision_at_50'] / res['baseline']['baseline_precision_at_50']
    print(f"\nThe learned model got {ratio:.1f}x more of its top-50 picks right than the rule —")
    print("measured evidence that this pattern is too tangled for a simple if-statement.")
else:
    print("Re-run scripts/run_all.py (see notebook 01) to regenerate outputs/model_results.json.")

Hand-written rule  Precision@50: 0.240
Random forest       Precision@50: 0.740

The learned model got 3.1x more of its top-50 picks right than the rule —
measured evidence that this pattern is too tangled for a simple if-statement.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.